# 04 — CVE Enrichment

**Goal:** Join the OSV vulnerability data onto our Sentinel Score table so each package has a `has_cve` flag and a list of known CVE IDs. This transforms our structural risk score into an actionable security report.

## What this notebook produces
| Output | Contents |
|---|---|
| `processed/sentinel_enriched.parquet` | Sentinel Score table with CVE flags added |
| `processed/blast_radius.parquet` | For each CVE-affected package, its downstream dependents ranked by exposure |

## Learning note: Version range matching
The hardest part of CVE enrichment is answering: "Is the *currently used* version of package X affected by this CVE?" OSV uses SEMVER ranges (`>=0.0.0, <4.17.21`) to express which versions are vulnerable. Matching a package's current version against these ranges requires a semver parser.

For this project, we take a **conservative approach**: if a package has *any* CVE, we flag it. We note in the report that in production you'd do exact version range matching (the `semver` Python library handles this).

## 0 — Setup

In [ ]:
!pip install -q pandas pyarrow semver

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd

BASE    = '/content/drive/MyDrive/BlastRadius'
OUT_DIR = f'{BASE}/data/processed'

# Load the data produced by previous notebooks
sentinel_pd = pd.read_parquet(f'{OUT_DIR}/sentinel_scores.parquet')
osv_pd      = pd.read_parquet(f'{OUT_DIR}/osv_npm.parquet')

print(f'Sentinel scores: {len(sentinel_pd):,} packages')
print(f'OSV records:     {len(osv_pd):,}')

## 1 — Join CVEs onto packages

**Strategy:** Match by package name (case-insensitive). Then, for packages that match, group all CVE IDs into a list. We add three columns to the Sentinel table:
- `has_cve` — boolean flag
- `cve_ids` — list of vulnerability IDs (GHSA-xxxx or CVE-xxxx)
- `max_severity` — worst severity among all CVEs for this package
- `cve_count` — number of distinct vulnerabilities

In [ ]:
# Aggregate OSV records per package: one row per package with all CVE IDs and worst severity
SEVERITY_ORDER = {'CRITICAL': 4, 'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, 'UNKNOWN': 0}

def worst_severity(sev_series):
    return max(sev_series, key=lambda s: SEVERITY_ORDER.get(s, 0))

cve_per_pkg = (
    osv_pd
    .groupby('package_name')
    .agg(
        cve_ids=('vuln_id', list),
        cve_count=('vuln_id', 'nunique'),
        max_severity=('severity', worst_severity),
        latest_cve_date=('published', 'max')
    )
    .reset_index()
)
cve_per_pkg['has_cve'] = True

print(f'Unique packages with CVEs: {len(cve_per_pkg):,}')
print(f'\nSeverity distribution of worst CVE per package:')
print(cve_per_pkg['max_severity'].value_counts())

In [ ]:
# Merge CVE data onto Sentinel scores
sentinel_enriched = (
    sentinel_pd
    .merge(
        cve_per_pkg.rename(columns={'package_name': 'name'}),
        on='name',
        how='left'
    )
)

# Fill defaults for packages with no CVEs
sentinel_enriched['has_cve']      = sentinel_enriched['has_cve'].fillna(False)
sentinel_enriched['cve_count']    = sentinel_enriched['cve_count'].fillna(0).astype(int)
sentinel_enriched['max_severity'] = sentinel_enriched['max_severity'].fillna('NONE')
sentinel_enriched['cve_ids']      = sentinel_enriched['cve_ids'].apply(
    lambda x: x if isinstance(x, list) else []
)

cve_pkg_count = sentinel_enriched['has_cve'].sum()
print(f'Packages with CVEs (in our graph): {cve_pkg_count:,} / {len(sentinel_enriched):,}')
print(f'\nTop 10 CVE-affected packages by Sentinel Score:')
print(
    sentinel_enriched[sentinel_enriched['has_cve']]
    .nlargest(10, 'sentinel_score')
    [['name', 'sentinel_score', 'has_cve', 'cve_count', 'max_severity', 'risk_tier']]
    .to_string(index=False)
)

In [ ]:
# Add a combined 'critical_flag': high Sentinel Score AND has a CVE
# These are the packages that need urgent attention
high_sentinel_threshold = sentinel_enriched['sentinel_score'].quantile(0.9)  # top 10%

sentinel_enriched['is_critical'] = (
    sentinel_enriched['has_cve'] &
    (sentinel_enriched['sentinel_score'] >= high_sentinel_threshold)
)

print(f'CRITICAL packages (high Sentinel Score + CVE): {sentinel_enriched["is_critical"].sum():,}')
print()
print('CRITICAL packages:')
print(
    sentinel_enriched[sentinel_enriched['is_critical']]
    .sort_values('sentinel_score', ascending=False)
    [['name', 'sentinel_score', 'bus_factor', 'cve_count', 'max_severity']]
    .to_string(index=False)
)

In [ ]:
# Save enriched table (convert list column to JSON string for parquet)
sentinel_save = sentinel_enriched.copy()
sentinel_save['cve_ids'] = sentinel_save['cve_ids'].apply(json.dumps)
sentinel_save.to_parquet(f'{OUT_DIR}/sentinel_enriched.parquet', index=False)
print(f'Saved enriched sentinel table → {OUT_DIR}/sentinel_enriched.parquet')

## 2 — Blast Radius: downstream impact of each CVE-affected package

**Why:** The Sentinel Score tells us which packages are *structurally fragile*. The blast radius tells us *what breaks* if one of them is compromised. These are different questions:
- A package can have high Sentinel Score but small blast radius (few downstream dependents)
- A package can have low Sentinel Score but massive blast radius (many dependents, but well-maintained)

The most dangerous packages are those at the intersection: high Sentinel Score AND large blast radius.

**Method:** For each CVE-affected package, we do a BFS (breadth-first search) through the dependency graph to find all packages that transitively depend on it. We cap at 3 hops to keep the computation tractable — in practice, 3 hops covers most meaningful exposure.

We use pure pandas/networkx for this (rather than Spark) since we're working on the top 500 packages.

In [ ]:
!pip install -q networkx

In [ ]:
import networkx as nx

# Load the dependency edge list
USE_SAMPLE = True
suffix = '_10k' if USE_SAMPLE else ''
DATA_DIR = f'{BASE}/data/sample' if USE_SAMPLE else f'{BASE}/data/processed'

edges_pd = pd.read_parquet(f'{DATA_DIR}/graph_edges{suffix}.parquet')
dep_edges = edges_pd[edges_pd['edge_type'] == 'DEPENDS_ON'][['src', 'dst']]

# Build a NetworkX directed graph
# Edge direction: src DEPENDS_ON dst
# For blast radius, we want: who depends on X? → follow REVERSE edges from X
G = nx.DiGraph()
G.add_edges_from(zip(dep_edges['src'], dep_edges['dst']))

print(f'NetworkX graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

In [ ]:
def blast_radius(G, package_id, max_hops=3):
    """
    Find all packages that transitively depend on `package_id`.
    Returns a list of (dependent_id, hop_distance) tuples.
    
    We traverse the REVERSED graph: in the original graph, 
    edges go src→dst meaning 'src depends on dst'.
    In the reversed graph, following edges from X gives us
    everything that depends on X.
    """
    G_rev = G.reverse(copy=False)
    if package_id not in G_rev:
        return []
    
    # BFS up to max_hops
    visited = {}
    queue = [(package_id, 0)]
    while queue:
        node, depth = queue.pop(0)
        if node in visited or depth > max_hops:
            continue
        if node != package_id:
            visited[node] = depth
        for neighbor in G_rev.neighbors(node):
            if neighbor not in visited:
                queue.append((neighbor, depth + 1))
    
    return [(node, hop) for node, hop in visited.items()]

# Compute blast radius for top 100 CVE-affected packages by Sentinel Score
cve_packages = (
    sentinel_enriched[sentinel_enriched['has_cve']]
    .nlargest(100, 'sentinel_score')
    ['name']
    .tolist()
)

blast_records = []
for pkg_name in cve_packages:
    pkg_id = f'pkg:{pkg_name.lower()}'
    dependents = blast_radius(G, pkg_id, max_hops=3)
    blast_records.append({
        'source_package': pkg_name,
        'source_id': pkg_id,
        'blast_radius_count': len(dependents),
        'direct_dependents': sum(1 for _, h in dependents if h == 1),
        'two_hop_dependents': sum(1 for _, h in dependents if h == 2),
        'three_hop_dependents': sum(1 for _, h in dependents if h == 3),
    })

blast_df = pd.DataFrame(blast_records)
blast_df = blast_df.merge(
    sentinel_enriched[['name', 'sentinel_score', 'bus_factor', 'cve_count', 'max_severity']],
    left_on='source_package', right_on='name', how='left'
).drop(columns='name').sort_values('blast_radius_count', ascending=False)

print('Top 15 CVE-affected packages by blast radius (3-hop downstream dependents):')
print(blast_df.head(15)[[
    'source_package', 'blast_radius_count', 'direct_dependents',
    'sentinel_score', 'max_severity'
]].to_string(index=False))

In [ ]:
blast_df.to_parquet(f'{OUT_DIR}/blast_radius.parquet', index=False)
print(f'Saved blast radius data → {OUT_DIR}/blast_radius.parquet')

## 3 — Case study: Visualize one package's blast radius

Pick the highest-Sentinel-Score CVE package and show its full downstream subgraph.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Pick the top package
top_cve_pkg = (
    sentinel_enriched[sentinel_enriched['has_cve']]
    .nlargest(1, 'sentinel_score')
    .iloc[0]
)
pkg_id   = f"pkg:{top_cve_pkg['name'].lower()}"
pkg_name = top_cve_pkg['name']

print(f'Case study package: {pkg_name}')
print(f'  Sentinel Score:  {top_cve_pkg["sentinel_score"]:.4f}')
print(f'  Bus Factor:      {top_cve_pkg["bus_factor"]}')
print(f'  CVE IDs:         {json.loads(top_cve_pkg["cve_ids"]) if isinstance(top_cve_pkg["cve_ids"], str) else top_cve_pkg["cve_ids"]}')

# Get blast radius (2 hops for readability)
dependents = blast_radius(G, pkg_id, max_hops=2)
print(f'\n2-hop blast radius: {len(dependents)} packages affected')

In [ ]:
# Build subgraph for visualization
blast_nodes = {pkg_id} | {node for node, _ in dependents}
subgraph    = G.reverse().subgraph(blast_nodes)

# Limit to top 50 nodes by degree for a readable plot
if len(subgraph.nodes) > 50:
    top_nodes = sorted(subgraph.nodes, key=lambda n: subgraph.degree(n), reverse=True)[:50]
    subgraph  = subgraph.subgraph(top_nodes)

# Node colors: red = source (vulnerable), orange = 1-hop, yellow = 2-hop
hop_map   = {node: hop for node, hop in dependents}
node_colors = [
    '#d62728' if n == pkg_id else ('#ff7f0e' if hop_map.get(n, 0) == 1 else '#ffbb78')
    for n in subgraph.nodes
]

# Node labels: strip the 'pkg:' prefix
labels = {n: n.replace('pkg:', '') for n in subgraph.nodes}

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(subgraph, seed=42, k=0.8)
nx.draw_networkx(
    subgraph, pos, ax=ax,
    labels=labels,
    node_color=node_colors,
    node_size=800,
    font_size=7,
    edge_color='#aaaaaa',
    arrows=True,
    arrowsize=10
)

patches = [
    mpatches.Patch(color='#d62728', label=f'Source: {pkg_name} (VULNERABLE)'),
    mpatches.Patch(color='#ff7f0e', label='1-hop dependents'),
    mpatches.Patch(color='#ffbb78', label='2-hop dependents'),
]
ax.legend(handles=patches, loc='upper left', fontsize=9)
ax.set_title(f'Blast Radius: {pkg_name} — {len(dependents)} affected packages (top 50 shown)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/blast_radius_{pkg_name}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved → {OUT_DIR}/blast_radius_{pkg_name}.png')

## 4 — Sanity checks

In [ ]:
# Check: lodash should have CVEs
lodash_row = sentinel_enriched[sentinel_enriched['name'] == 'lodash']
if len(lodash_row) > 0:
    r = lodash_row.iloc[0]
    print(f'CHECK — lodash: has_cve={r["has_cve"]}, cve_count={r["cve_count"]}, max_severity={r["max_severity"]}')
    if r['has_cve']:
        print('  lodash has CVEs ✓')
    else:
        print('  WARNING: lodash shows no CVEs — check OSV join')
else:
    print('lodash not found in sentinel table')

In [ ]:
print('Enrichment complete.')
print('Next: open 05_neo4j_load.ipynb')